# 原 Stage1 tail 输出与 SOXL−SOXX H5 相对收益：必要信息检验

原39输入预测头、冻结940个成熟 H5 标签；完整944日历区块抽样，不拟合、不跑账户。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from scipy.stats import spearmanr
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_tail_score_soxl_relative_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['source'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('external_requests','new_model_fits','new_native_accounts'))
state_path=card/'run_state.json';tracking=card/'qlib_recorder';assert not state_path.exists() and not tracking.exists() and not (card/'result.json').exists()
a=root/src['Stage1_card'];b=root/src['relative_label_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['Stage1_summary_sha256'] and sha(b/'summary.json')==src['relative_label_summary_sha256']
assert sa['recorder_id']==src['Stage1_recorder_id'] and sb['recorder_id']==src['relative_label_recorder_id']
assert sa['nas_evidence']['sha256']==src['Stage1_NAS_sha256'] and sb['nas_evidence']['sha256']==src['relative_label_NAS_sha256']
assert sha(a/'qlib_recorder'/src['Stage1_experiment_id']/src['Stage1_recorder_id']/'artifacts/inputs/frame')==src['Stage1_frame_sha256']
assert sha(b/'qlib_recorder'/src['relative_label_experiment_id']/src['relative_label_recorder_id']/'artifacts/analysis/matched_daily')==src['matched_daily_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['provider_uri']),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):frame=R.get_recorder(recorder_id=src['Stage1_recorder_id'],experiment_id=src['Stage1_experiment_id']).load_object('inputs/frame')
with R.uri_context(str(b/'qlib_recorder')):matched=R.get_recorder(recorder_id=src['relative_label_recorder_id'],experiment_id=src['relative_label_experiment_id']).load_object('analysis/matched_daily')
cal=pd.DatetimeIndex(D.calendar(start_time=plan['cohort']['full_signal'][0],end_time=plan['cohort']['full_signal'][1],freq='day'))
assert len(cal)==len(frame)==len(matched)==944 and cal.equals(pd.DatetimeIndex(pd.to_datetime(frame.signal_date))) and cal.equals(pd.DatetimeIndex(pd.to_datetime(matched.signal_date)))
assert pd.DatetimeIndex(pd.to_datetime(frame.trade_date)).equals(pd.DatetimeIndex(pd.to_datetime(matched.trade_date)))
assert (pd.to_datetime(frame.feature_available_at)<pd.to_datetime(frame.decision_at)).all()
assert matched.H1_mature.sum()==944 and matched.H5_mature.sum()==940 and matched.H5_mature.iloc[:-4].all() and (~matched.H5_mature.iloc[-4:]).all()
assert (pd.to_datetime(frame.decision_at.loc[matched.H5_mature],utc=True)<pd.to_datetime(matched.H5_available_at.loc[matched.H5_mature],utc=True)).all()
d=pd.DataFrame({'signal_date':cal,'trade_date':pd.to_datetime(frame.trade_date),'tail_probability':frame.tail_probability.astype(float),'relative_H1':matched.relative_H1.astype(float),'relative_H5':matched.relative_H5.astype(float),'H1_mature':matched.H1_mature.astype(bool),'H5_mature':matched.H5_mature.astype(bool),'H5_available_at':matched.H5_available_at})
assert np.isfinite(d.tail_probability).all() and d.tail_probability.between(0,1).all()
static=pd.concat({'feature':d[['tail_probability']],'label':d[['relative_H1','relative_H5']],'meta':d[['H1_mature','H5_mature']]},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['SOXL_SOXX_CASH']*len(cal)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':tuple(plan['cohort']['full_signal'])});assert len(dataset.prepare('held',col_set=['feature','label','meta']))==944
rho=lambda z,col:float(spearmanr(z.tail_probability,z[col]).statistic) if len(z)>10 and z.tail_probability.nunique()>1 else None
h5=d.loc[d.H5_mature];h1=d.loc[d.H1_mature];primary=rho(h5,'relative_H5');secondary=rho(h1,'relative_H1');assert primary is not None and secondary is not None
half1=h5.loc[h5.signal_date<pd.Timestamp('2025-01-01')];half2=h5.loc[h5.signal_date>=pd.Timestamp('2025-01-01')];halves=[rho(half1,'relative_H5'),rho(half2,'relative_H5')];assert min(len(half1),len(half2))>300
annual={str(y):{'rows':len(g),'spearman':rho(g,'relative_H5'),'mean_tail_probability':float(g.tail_probability.mean()),'mean_relative_H5':float(g.relative_H5.mean())} for y,g in h5.groupby(h5.signal_date.dt.year)}
quarters={str(q):{'rows':len(g),'spearman':rho(g,'relative_H5'),'mean_tail_probability':float(g.tail_probability.mean()),'mean_relative_H5':float(g.relative_H5.mean())} for q,g in h5.groupby(h5.signal_date.dt.to_period('Q'))};assert len(quarters)==16
cfg=plan['statistics']['uncertainty'];state={'plan_sha256':sha(card/'plan.json'),'bootstrap_attempted':True,'bootstrap_draw_budget':cfg['draws'],'model_fits':0,'native_accounts':0};state_path.write_text(json.dumps(state,indent=2)+'\n')
rng=np.random.default_rng(cfg['seed']);n=len(d);replicates=np.empty(cfg['draws'],float)
for bidx in range(cfg['draws']):
 idx=np.empty(n,dtype=np.int32);idx[0]=rng.integers(n)
 for j in range(1,n):idx[j]=rng.integers(n) if rng.random()<1/cfg['mean_block_sessions'] else (idx[j-1]+1)%n
 sample=d.iloc[idx];sample=sample.loc[sample.H5_mature & sample.relative_H5.notna()];replicates[bidx]=rho(sample,'relative_H5')
assert np.isfinite(replicates).all();ci=np.quantile(replicates,cfg['two_sided_descriptive_quantiles']);support=bool(primary<0 and all(x is not None and x<0 for x in halves) and all(annual[str(y)]['spearman']<0 for y in plan['cohort']['complete_years']) and ci[1]<0)
report={'decision':'conditional_tail_score_SOXL_relative_H5_necessary_information_support_development_only' if support else 'closed_fixed_tail_score_SOXL_relative_H5_necessary_information_failed','primary_H5_spearman':primary,'secondary_H1_spearman':secondary,'held_halves_H5_spearman':halves,'complete_2024_2025_H5_spearman':{str(y):annual[str(y)]['spearman'] for y in (2024,2025)},'descriptive_95pct_full_calendar_block_interval':[float(ci[0]),float(ci[1])],'necessary_support':support,'annual':annual,'quarters':quarters,'calendar_rows':944,'mature_H5_rows':940,'bootstrap_replicates':len(replicates),'source_new_fits':0,'new_native_accounts':0,'historical_independence':False,'this_is_project_wide_multiple_test_corrected':False}
assert state['bootstrap_attempted'] and not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='analysis',matched_daily=d,report=report,bootstrap_replicates=replicates);rec.log_metrics(primary_H5_spearman=primary,block_upper=float(ci[1]),necessary_support=int(support),model_fits=0,native_accounts=0)
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Same reused 2023-2026 history already searched repeatedly, descriptive interval is not project-wide corrected confirmation.','Gross original price-plane H5 relative return is not complete fee-net account result.','Even positive information is only a necessary gate, not a Stage1 tail-policy or SOXX success.']}
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n');(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n');e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='tail-relative-info-restore-') as t:
 archive=Path(t)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/matched_daily') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state.update(completed=True,recorder_id=rid,NAS_sha256=e['sha256']);state_path.write_text(json.dumps(state,indent=2)+'\n');print({k:report[k] for k in ('decision','primary_H5_spearman','held_halves_H5_spearman','complete_2024_2025_H5_spearman','descriptive_95pct_full_calendar_block_interval')})


2026/10/23 07:18:09 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[15903:MainThread](2026-10-23 07:18:09,964) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[15903:MainThread](2026-10-23 07:18:09,966) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[15903:MainThread](2026-10-23 07:18:09,966) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


...

[15903:MainThread](2026-10-23 07:18:11,376) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_tail_score_soxl_relative_information_20261023.


[15903:MainThread](2026-10-23 07:18:11,378) INFO - qlib.workflow - [exp.py:258] - Experiment 379411153587244423 starts running ...


[15903:MainThread](2026-10-23 07:18:11,463) INFO - qlib.workflow - [recorder.py:345] - Recorder 8b6f1604b17d4f3ebe6463ab54745f9a starts running under Experiment 379411153587244423 ...


[15903:MainThread](2026-10-23 07:18:12,510) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{'decision': 'closed_fixed_tail_score_SOXL_relative_H5_necessary_information_failed', 'primary_H5_spearman': 0.030751616822043058, 'held_halves_H5_spearman': [0.09252531664511109, -0.01339915580227239], 'complete_2024_2025_H5_spearman': {'2024': 0.07081030292532112, '2025': -0.06221245139922238}, 'descriptive_95pct_full_calendar_block_interval': [-0.07372297093100852, 0.14204192829668877]}


## 中文结论

# 原 Stage1 tail 输出对 SOXL 相对 SOXX：固定必要信息门失败

冻结原始 Stage1 的 `tail_probability`，与已保存的五个开盘持有区间 SOXL−SOXX 相对对数收益逐信号日配对。完整 Qlib 日历 944 日、成熟 H5 标签 940 日、末端 4 日未知；一次 63 日均长/2,000 次的配对区块抽样先在完整日历取索引再应用成熟掩码。没有重新拟合或资金账户。

预定机制要求 tail 分数越高，后续 SOXL 相对收益越低。实际全期 H5 Spearman **+0.0308**，方向相反；H1 为 +0.0220。H5 前后两半 +0.0925/-0.0134，完整 2024/2025 年 +0.0708/-0.0622，描述性 95% 区块区间 [-0.0737, +0.1420] 跨零。四项必要支持没有同时成立，决定 `closed_fixed_tail_score_SOXL_relative_H5_necessary_information_failed`。2026 年或个别亏损日的高 tail 值不能救援全期负向假说，不调整符号、持有期、年份或门槛。

原生 Qlib Recorder `8b6f1604b17d4f3ebe6463ab54745f9a` 中的完整944日、2000次复制和指标已定向读回；Yanxuan NAS SHA-256 `e33d026cb65a6433496b79a4672fb76a1009c1c84be6c3d93fe4b471b59af9d7` 经远端哈希及限定恢复，独立复核确认失败。历史 2023–2026 已重复使用，区间未作全项目研究选择校正；信息排序也不是 Tiered 费用后的完整自有状态账户。**不执行 tail 输出直接路由的 0bp 账户**。整体稳定超过 SOXX、2/5bp 与独立前向证据仍未达成，下一机制需带来不同且决策时可用的信息。
